####1. Using Transformer LLMs for Zero and Few Shot Learning

#### ZERO SHOT PROMPTING
* Definition: <br>
You ask the model to perform a task without giving any examples.

The model relies only on: <br>
its pre-training knowledge <br>
the instruction in the promp

#### FEW SHOT PROMPTING
* Definition : <br>
You give the model a few examples of the task before asking it to solve a new one.The model learns the pattern from examples in the prompt itself.



1.1 Create a pipeline for text-generation using a model Qwen/Qwen2.5-1.5B-Instruct

In [0]:
from transformers import pipeline, set_seed
pipe = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct")
set_seed(45)

2026-03-05 09:37:49.548527: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-05 09:37:49.563039: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-05 09:37:49.652338: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-05 09:37:49.745739: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772703469.823537    1907 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772703469.84

[2026-03-05 09:37:58,310] [WARNING] [real_accelerator.py:194:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.
[2026-03-05 09:37:58,319] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cpu (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


1.2 Try a Zero-shot generation for the given requirement.

In [0]:
instruction = """Given a hobby of a person, suggest a description of that person.
The description should not include the original hobby."""

io_string = """
hobby: painting
description:"""

prompt = instruction + io_string
print(prompt)

Given a hobby of a person, suggest a description of that person.
The description should not include the original hobby.
hobby: painting
description:


1.3 Generate the result

In [0]:
result=pipe(prompt, min_new_tokens=5, max_new_tokens=20)
print(result[0]["generated_text"])

Given a hobby of a person, suggest a description of that person.
The description should not include the original hobby.
hobby: painting
description: creative artist with an eye for detail and color. The person has a unique style in their paintings,


1.4 Create a few-shot prompt for the given requirement.

In [0]:
examples="""
word: happy
description: smiling, laughing, clapping

word: nervous
description: glancing around quickly, sweating, fidgeting

word: sleepy
description: heavy-lidded, slumping, rubbing eyes
"""
prompt = instruction + examples + io_string
print(prompt)

Given a hobby of a person, suggest a description of that person.
The description should not include the original hobby.
word: happy
description: smiling, laughing, clapping

word: nervous
description: glancing around quickly, sweating, fidgeting

word: sleepy
description: heavy-lidded, slumping, rubbing eyes

hobby: painting
description:


1.5 Generate the result

In [0]:
result=pipe(prompt, min_new_tokens=5, max_new_tokens=20)
print(result[0]["generated_text"])

Given a hobby of a person, suggest a description of that person.
The description should not include the original hobby.
word: happy
description: smiling, laughing, clapping

word: nervous
description: glancing around quickly, sweating, fidgeting

word: sleepy
description: heavy-lidded, slumping, rubbing eyes

hobby: painting
description: sketching, experimenting with colors, mixing paints Description:
sketching, experimenting with colors, mixing


1.6 Improve the results of few-shot generation with the help of stop sequence

In [0]:
examples="""
word: happy
description: smiling, laughing, clapping
###
word: nervous
description: glancing around quickly, sweating, fidgeting
###
word: sleepy
description: heavy-lidded, slumping, rubbing eyes
###
"""
prompt = instruction + examples + io_string
print(prompt)

Given a hobby of a person, suggest a description of that person.
The description should not include the original hobby.
word: happy
description: smiling, laughing, clapping
###
word: nervous
description: glancing around quickly, sweating, fidgeting
###
word: sleepy
description: heavy-lidded, slumping, rubbing eyes
###

hobby: painting
description:


1.7 Generate the result

In [0]:
eos_token_id = pipe.tokenizer.encode("###")[0]
result=pipe(prompt, min_new_tokens=5, max_new_tokens=20, eos_token_id=eos_token_id)
print(result[0]["generated_text"])

Given a hobby of a person, suggest a description of that person.
The description should not include the original hobby.
word: happy
description: smiling, laughing, clapping
###
word: nervous
description: glancing around quickly, sweating, fidgeting
###
word: sleepy
description: heavy-lidded, slumping, rubbing eyes
###

hobby: painting
description: focused gaze, mixing colors, experimenting with brushes
###
